# Συνοδευτικό notebook κεφαλαίου 17
**Φτιάξε το πρώτο σου μεγάλο γλωσσικό μοντέλο — Κεφάλαιο 17: Επιλογές ανάπτυξης στο νέφος**

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch17.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch17.ipynb). Τα κελιά εμφανίζουν παραδείγματα αρχείων και εντολές για το τερματικό. Δεν χρειάζονται διαπιστευτήρια ή βοηθητικό αρχείο, δεν στέλνουν αιτήματα και δεν αναπτύσσουν καμία υπηρεσία. Οι προτροπές, οι τιμές του API και οι συμβολοσειρές που συγκρίνονται με αναμενόμενες αποκρίσεις παραμένουν στα αγγλικά.

Αν επιλέξεις να εκτελέσεις τα παραδείγματα στο Modal, χρησιμοποίησε το δικό σου τοπικό τερματικό. Η κλήση απομακρυσμένων συναρτήσεων μπορεί να επιφέρει χρεώσεις· η εκτέλεση των κελιών από μόνη της δεν μπορεί να θέσει σε λειτουργία GPU.

In [ ]:
# Τα παρακάτω κελιά εμφανίζουν κώδικα· μην εγκαταστήσεις τίποτα στο περιβάλλον του notebook.
print('Προαιρετική ρύθμιση στο τοπικό τερματικό (χρησιμοποίησε εικονικό περιβάλλον με Python 3.11):')
print('python -m pip install modal fastapi pydantic requests')
print('modal setup  # Πιστοποίηση του CLI· δεν πρόκειται για διακριτικό του διαμεσολαβητή ιστού.')

## Ενότητα 17.1: Γιατί Modal;

**Τι σημαίνει serverless;**

| Συμβατικός εξυπηρετητής | Χωρίς διαχείριση εξυπηρετητή (serverless, Modal) |
|-------------------|-------------------|
| Νοικιάζεις έναν υπολογιστή όλο το εικοσιτετράωρο | Ο κώδικας εκτελείται μόνο όταν χρειάζεται |
| Πληρώνεις ακόμη και όταν ο υπολογιστής είναι αδρανής | Πληρώνεις ανά δευτερόλεπτο υπολογιστικού χρόνου |
| Διαχειρίζεσαι το μηχάνημα | Το Modal διαχειρίζεται το μηχάνημα· εσύ εξακολουθείς να ελέγχεις την πρόσβαση στην εφαρμογή |
| Σταθερή δυναμικότητα | Αυτόματη κλιμάκωση |

**Αναλογία:** Η συμβατική φιλοξενία μοιάζει με το να έχεις δικό σου αυτοκίνητο. Το serverless μοιάζει με ταξί: πληρώνεις μόνο για τη διαδρομή.

**Γιατί να χρησιμοποιήσεις το Modal για LLM;**
- Ενσωματωμένη υποστήριξη GPU (από T4 έως H100)
- Δωρεάν πιστώσεις 30 δολαρίων τον μήνα (περίπου 37 ώρες σε L4 ή 50 ώρες σε T4, με τις τιμές του παραδείγματος)
- Ρύθμιση σε Python, χωρίς Docker ή YAML
- Τα απομονωμένα περιβάλλοντα εκτέλεσης (container) μπορούν να μειωθούν μέχρι το μηδέν. Η χρέωση για χρόνο GPU παύει μόλις σταματήσει και το τελευταίο container.

## Ενότητα 17.2: Πρώτη γνωριμία με το Modal

Αυτή είναι η απλούστερη εφαρμογή Modal. Αποθήκευσε τον κώδικα ως `hello.py` και εκτέλεσε `modal run hello.py`:

In [ ]:
# Αποθήκευσε τον κώδικα ως hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Αποθήκευσε τον κώδικα ως 'hello.py':")
print(hello_modal_code)
print("\nΈπειτα εκτέλεσε: modal run hello.py")

**Τι συμβαίνει όταν εκτελείς την εντολή;**

1. Η `modal.App()` δημιουργεί την εφαρμογή σου.
2. Με το `@app.function()` δηλώνεις ποιος κώδικας θα εκτελεστεί στο νέφος του Modal.
3. Η `hello.remote()` καλεί τη συνάρτηση απομακρυσμένα.
4. Το Modal ξεκινά ένα container, εκτελεί τον κώδικα και επιστρέφει το αποτέλεσμα.

## Ενότητα 17.3: Προσθήκη υποστήριξης GPU

Αρκεί μία παράμετρος για να προσθέσεις υποστήριξη GPU:

In [ ]:
# Αποθήκευσε τον κώδικα ως gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# Η εικόνα του container πρέπει να περιέχει το torch (θα δούμε τις εικόνες στην επόμενη ενότητα)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # Αυτό αρκεί!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Αποθήκευσε τον κώδικα ως 'gpu_test.py':")
print(gpu_test_code)
print("\nΈπειτα εκτέλεσε: modal run gpu_test.py")
print("\nΑναμενόμενη έξοδος: GPU available: NVIDIA L4")

**Μία μόνο παράμετρος.** Δεν χρειάζεται να εγκαταστήσεις CUDA, να διαχειριστείς οδηγούς ή να φτιάξεις εικόνες Docker. Το Modal αναλαμβάνει όλο το λογισμικό για τη GPU.

## Ενότητα 17.4: Εγκατάσταση εξαρτήσεων

Το μοντέλο σου χρειάζεται βιβλιοθήκες. Με το Modal ορίζεις το περιβάλλον του container σε Python:

In [ ]:
# Αποθήκευσε τον κώδικα ως with_deps.py
with_deps_code = '''
import modal

# Όρισε την εικόνα του container
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Αποθήκευσε τον κώδικα ως 'with_deps.py':")
print(with_deps_code)
print("\nΈπειτα εκτέλεσε: modal run with_deps.py")

Η παράμετρος `image` δηλώνει στο Modal τι να εγκαταστήσει. Η πρώτη εκτέλεση διαρκεί περισσότερο, επειδή τότε δημιουργείται η εικόνα. Οι επόμενες εκτελέσεις χρησιμοποιούν την ήδη αποθηκευμένη εικόνα.

## Ενότητα 17.5: Τελικά σημεία ιστού με το FastAPI

Το παράδειγμα χρησιμοποιεί μόνο CPU και έχει δύο διαδρομές: η `/health` επιστρέφει μια σταθερή ένδειξη κατάστασης και η `/hello/{name}` έναν χαιρετισμό. Προστάτευσε ολόκληρη την εφαρμογή ASGI, μαζί με την τεκμηρίωσή της, ενεργοποιώντας τον έλεγχο ταυτότητας στον διαμεσολαβητή του Modal. Η διεύθυνση URL προσδιορίζει την υπηρεσία· δεν δίνει άδεια χρήσης.

Πριν δοκιμάσεις κάποιο από τα δύο παραδείγματα ιστού, δημιούργησε ένα **διακριτικό διαμεσολαβητή (proxy token)** στο Modal από το Settings → Proxy auth tokens (ή εκτέλεσε `modal workspace proxy-tokens create --name chapter17` σε ιδιωτικό τερματικό). Φύλαξε το αναγνωριστικό `wk-…` και το μυστικό `ws-…`, που εμφανίζεται μόνο μία φορά, στον διαχειριστή κωδικών σου. Χρησιμοποίησε ξεχωριστό περιβάλλον επίδειξης. Αν ο χώρος εργασίας υποστηρίζει έλεγχο πρόσβασης βάσει ρόλων (RBAC), δώσε στο διακριτικό πρόσβαση μόνο σε αυτό το περιβάλλον. Το όνομα της εφαρμογής ως όνομα του διακριτικού δεν το περιορίζει σε μία διεύθυνση URL. Εμπιστεύσου το μόνο σε εξουσιοδοτημένους χειριστές· μην το βάλεις σε notebook, σε κώδικα που εκτελείται στον φυλλομετρητή, σε στιγμιότυπο οθόνης ή σε αποθετήριο.

In [ ]:
# Αποθήκευσε τον κώδικα ως simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

# Η υπηρεσία ιστού εκτελείται ΜΕΣΑ στο container του Modal, όχι στον υπολογιστή σου.
# Η εικόνα πρέπει να περιέχει τις εξαρτήσεις της στις εκδόσεις του δοκιμασμένου περιβάλλοντος του βιβλίου:
# fastapi και pydantic για την εφαρμογή, uvicorn για τον εξυπηρετητή ASGI και httpx
# για ελέγχους τύπου TestClient μέσα στο container.
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2",
    "uvicorn==0.54.0",
    "httpx==0.28.1",
    "pydantic==2.14.3",
)

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function(image=image)
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    return web_app
'''

print("Αποθήκευσε τον κώδικα ως 'simple_api.py':")
print(simple_api_code)
print("\nΈπειτα εκτέλεσε: modal deploy simple_api.py")
print("\nΘα λάβεις μια διεύθυνση URL όπως: https://your-workspace--my-api-serve.modal.run")

**Προαιρετικός έλεγχος στο τερματικό.** Σε ιδιωτική συνεδρία Bash, όρισε ως `SIMPLE_URL` την ακριβή διεύθυνση προέλευσης HTTPS που εμφάνισε το Modal για τη **my-api**. Φόρτωσε από τον διαχειριστή μυστικών τα διαπιστευτήρια του διαμεσολαβητή στις `MODAL_KEY` και `MODAL_SECRET` της συνεδρίας. Μην πληκτρολογήσεις μυστικές τιμές σε εντολές που αποθηκεύονται στο ιστορικό και μην ενεργοποιήσεις την καταγραφή εκτελούμενων εντολών (shell tracing).

```bash
curl -i --max-time 30 "$SIMPLE_URL/health"  # αναμένεται 401 από τον διαμεσολαβητή
curl -i --max-time 30 --header @- "$SIMPLE_URL/health" <<EOF
Modal-Key: $MODAL_KEY
Modal-Secret: $MODAL_SECRET
EOF
# Αναμένεται: {"status":"healthy"}. Η /hello/World χρειάζεται τις ίδιες δύο κεφαλίδες.
modal app stop my-api
```

Ακόμη κι αν αποτύχει κάποιος έλεγχος, σταμάτησε τη `my-api` και βεβαιώσου από τον πίνακα ελέγχου ότι δεν απομένουν ενεργά container. Μια απλή επίσκεψη από φυλλομετρητή δεν αποτελεί επιτυχή έλεγχο: το Modal απορρίπτει το αίτημα πριν εκτελεστεί η συνάρτηση.

## Ενότητα 17.6: Η πλήρης υπηρεσία LLM

Η υπηρεσία GPU εκτελεί το προεκπαιδευμένο Qwen3.5-2B σε μια L4. Όπως στο παράδειγμα με CPU, το Modal ελέγχει το ζεύγος διαπιστευτηρίων του διαμεσολαβητή **πριν** ξεκινήσει ή επανενεργοποιήσει ένα container. Η `/health` ελέγχει αν το vLLM είναι έτοιμο, χωρίς να παράγει κείμενο. Η `/chat` ελέγχει επιπλέον ένα ξεχωριστό διακριτικό bearer της εφαρμογής **μέσα** στο FastAPI, πριν εκτελεστεί το μοντέλο. Όποιος διαθέτει έγκυρο ζεύγος για τον διαμεσολαβητή μπορεί να ενεργοποιήσει μια GPU, ακόμη κι αν η `/chat` επιστρέψει στη συνέχεια 401.

In [ ]:
# Αποθήκευσε τον κώδικα ως llm_service.py
# Εμφανίζεται μόνο ο κώδικας της υπηρεσίας. Ο τοπικός εξυπηρετητής vLLM δεν είναι δεύτερο τελικό σημείο ιστού.
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# Δεν χρειάζεται trust_remote_code. Το vLLM υποστηρίζει εγγενώς σύγχρονα μοντέλα όπως το Qwen3.5.
# Ενεργοποίησε την επιλογή μόνο αν έχεις ελέγξει τον ειδικό κώδικα του αποθετηρίου,
# καθώς η επιλογή επιτρέπει την εκτέλεση κώδικα από το αποθετήριο του μοντέλου.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Σταθερή έκδοση για αναπαραγωγιμότητα (σταθερή κατά τον έλεγχο του Νοεμβρίου 2026)
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="Το δικό μου API για LLM", version="1.0.0")

# Η δήλωση του σχήματος bearer εμφανίζει το κουμπί «Authorize» στο Swagger και επισημαίνει
# τις προστατευμένες διαδρομές στην τεκμηρίωση· τον πραγματικό έλεγχο τον κάνει η require_token.
bearer_scheme = HTTPBearer(auto_error=False)

# ===== Μέτρα προστασίας του κεφαλαίου 16, έλεγχος ταυτότητας και όρια token =====
MAX_PROMPT_CHARS = 4000  # γρήγορο πρώτο φίλτρο· το πραγματικό όριο μετριέται σε token
API_TOKEN = os.environ.get("API_TOKEN", "")  # ορίζεται μέσω μυστικού του Modal, βλ. παρακάτω
MAX_INPUT_TOKENS = 2048  # όριο εισόδου μετά την εφαρμογή του προτύπου (μαζί με την έξοδο χωρά στα 8.192 token)
MAX_OUTPUT_TOKENS = 512  # όριο παραγωγής ανά αίτημα, όχι όριο χρέωσης
RATE_LIMIT = 20
RATE_WINDOW = 60  # δευτερόλεπτα στο μονοτονικό ρολόι
request_counts: dict[str, list[float]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Έλεγχος ταυτότητας με διακριτικό bearer: ένα κοινόχρηστο τελικό σημείο GPU δεν πρέπει να είναι ανοιχτό σε όλους."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # Εφάρμοσε το όριο αντί να απορρίψεις το αίτημα: αν ζητηθούν περισσότερα token, δεν επιστρέφεται σφάλμα.
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = time.monotonic()
    # Έλεγχος και ενημέρωση με το ίδιο κλείδωμα: τα τελικά σημεία εκτελούνται σε νήματα του FastAPI.
    # Χωρίς κλείδωμα, δύο ταυτόχρονα αιτήματα θα μπορούσαν να διαβάσουν και τα δύο
    # len == RATE_LIMIT - 1 και να περάσουν μαζί.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Κατάγραψε μόνο ρητά επιτρεπόμενα μεταδεδομένα, ποτέ προτροπές, απαντήσεις ή το αρχικό
    # κείμενο εξαιρέσεων. Το Modal καταγράφει την τυπική έξοδο (stdout).
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# ===== Διαχείριση εξυπηρετητή vLLM (ένας ανά container) =====
_server = None
_tokenizer = None

def start_vllm():
    """Εκκίνησε το vllm serve μία φορά ανά container και περίμενε να είναι έτοιμο."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",            # οι πελάτες βλέπουν μόνο την εφαρμογή ASGI με έλεγχο ταυτότητας
            "--port", "8000",
            "--max-model-len", "8192",       # περιόρισε το μήκος συμφραζομένων ώστε η κρυφή μνήμη KV να μένει μικρή στα 24 GB της L4
            "--gpu-memory-utilization", "0.90",
            "--dtype", "bfloat16",           # βάρη Qwen3.5· η L4 υποστηρίζει bfloat16, η T4 όχι
        ])
        # Περίμενε να ανταποκριθεί το ελαφρύ τελικό σημείο /health του ίδιου του εξυπηρετητή
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Ελαφρύς έλεγχος: ρώτησε τον τοπικό εξυπηρετητή vLLM χωρίς παραγωγή κειμένου."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # 503, όχι 200: η παρακολούθηση πρέπει να δείχνει ότι η υποβαθμισμένη υπηρεσία δεν είναι διαθέσιμη
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Για να παραχθεί απάντηση, εφάρμοσε το πρότυπο στην προτροπή και άφησε το vLLM να εκτελέσει το μοντέλο."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.perf_counter()
    try:
        tokenizer = start_vllm()
        # Τα μοντέλα που έχουν εκπαιδευτεί να ακολουθούν οδηγίες περιμένουν το δικό τους πρότυπο συνομιλίας, όχι απλό κείμενο.
        # Η apply_chat_template πλαισιώνει το μήνυμα με τα ειδικά token του μοντέλου.
        messages = [{"role": "user", "content": request.message}]
        # Για το πραγματικό όριο μέτρησε την είσοδο μετά την εφαρμογή του προτύπου με τον ίδιο tokenizer
        # που χρησιμοποιεί το vLLM. Οι χαρακτήρες δεν είναι token. Για μία συνομιλία η
        # apply_chat_template επιστρέφει παρτίδα που συμπεριφέρεται σαν λεξικό.
        # Το πεδίο «input_ids» περιέχει την επίπεδη λίστα των token. Η len() στην ίδια την παρτίδα
        # θα μετρούσε τα κλειδιά (2), όχι τα token.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Προώθησε ακριβώς τα ίδια ID που μόλις μέτρησες. Η /v1/completions δέχεται
        # λίστα με ID token στο πεδίο «prompt», οπότε το vLLM δεν μπορεί να χωρίσει ξανά
        # το κείμενο σε διαφορετικά token από εκείνα που μέτρησες.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.perf_counter() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # Το ίδιο το vLLM απέρριψε το αίτημα (π.χ. υπέρβαση μήκους συμφραζομένων). Διατήρησε
        # τον κωδικό 4xx με ασφαλές μήνυμα, ποτέ με το αρχικό κείμενο του εξυπηρετητή.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.perf_counter() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # διατήρηση ενεργού container για 5 λεπτά
    volumes={"/root/.cache/huggingface": model_cache},  # το μοντέλο λαμβάνεται μία φορά
    secrets=[modal.Secret.from_name("llm-api-token")],  # παρέχει το API_TOKEN
)
@modal.concurrent(max_inputs=10)  # ασφαλές: ο εξυπηρετητής vLLM ομαδοποιεί τα ταυτόχρονα αιτήματα σε παρτίδες
@modal.asgi_app(requires_proxy_auth=True)  # το Modal ελέγχει την πρόσβαση πριν από την εκκίνηση της GPU
def serve():
    start_vllm()  # μία φόρτωση στην εκκίνηση του container για όλα τα αιτήματα
    return web_app

'''

print("Αποθήκευσε τον κώδικα ως 'llm_service.py':")
print(llm_service_code)

In [ ]:
# Αποθήκευσε τον κώδικα του πελάτη τοπικά ως check_service.py. Το notebook δεν στέλνει αιτήματα.
client_code = '''
import os
import requests

URL = os.environ["LLM_URL"].rstrip("/")  # η ακριβής διεύθυνση προέλευσης HTTPS από τη δική σου ανάπτυξη
if not URL.startswith("https://"):
    raise ValueError("Η LLM_URL πρέπει να χρησιμοποιεί HTTPS")
proxy = {"Modal-Key": os.environ["MODAL_KEY"], "Modal-Secret": os.environ["MODAL_SECRET"]}
chat = {**proxy, "Authorization": "Bearer " + os.environ["API_TOKEN"]}

# Έλεγξε πρώτα τον διαμεσολαβητή. Αυτό το αίτημα δεν πρέπει να ενεργοποιήσει GPU.
r = requests.get(f"{URL}/health", timeout=30, allow_redirects=False)
assert r.status_code == 401, f"Αναμενόταν απόρριψη από τον διαμεσολαβητή, αλλά ο κωδικός ήταν {r.status_code}"

r = requests.get(f"{URL}/health", headers=proxy, timeout=300, allow_redirects=False)
assert r.status_code == 200 and r.json()["status"] == "healthy", f"Έλεγχος ετοιμότητας: {r.status_code}"

def ask(headers, payload):
    return requests.post(f"{URL}/chat", headers=headers, json=payload,
                         timeout=150, allow_redirects=False)

r = ask(proxy, {"message": "hi"})  # η είσοδος μέσω διαμεσολαβητή δεν συνεπάγεται εξουσιοδότηση από την εφαρμογή
assert r.status_code == 401, f"Αναμενόταν απόρριψη από την εφαρμογή, αλλά ο κωδικός ήταν {r.status_code}"
r = ask({**proxy, "Authorization": "Bearer wrong-token"}, {"message": "hi"})
assert r.status_code == 401, f"Αναμενόταν απόρριψη του λανθασμένου bearer, αλλά ο κωδικός ήταν {r.status_code}"
r = ask(chat, {"message": " "})
assert r.status_code == 422, f"Αναμενόταν σφάλμα επικύρωσης, αλλά ο κωδικός ήταν {r.status_code}"
r = ask(chat, {"message": "What is Python?", "max_tokens": 64})
assert r.status_code == 200 and r.json()["response"].strip(), f"Συνομιλία: {r.status_code}"
print("Πέτυχαν οι έλεγχοι διαμεσολαβητή, ετοιμότητας, ταυτότητας στην εφαρμογή, επικύρωσης και συνομιλίας.")
'''
print(client_code)


### Τι να ελέγξεις στην πρώτη εκτέλεση

Για **προαιρετική δοκιμή της πραγματικής υπηρεσίας** στο τερματικό σου, όχι σε αυτό το notebook:

1. Στον πίνακα ελέγχου του Modal, επίλεξε το περιβάλλον προορισμού και δημιούργησε το μυστικό `llm-api-token` με κλειδί `API_TOKEN` και μεγάλη, τυχαία παραγόμενη τιμή. Μόνο η υπηρεσία GPU χρειάζεται αυτό το μυστικό. Το διακριτικό διαμεσολαβητή του χώρου εργασίας από την §17.5 είναι ξεχωριστό· μη χρησιμοποιήσεις τα διαπιστευτήρια του CLI σε κανένα από τα δύο επίπεδα.
2. Αποθήκευσε τον κώδικα που εμφανίζεται για την υπηρεσία και τον πελάτη. Όρισε πρώτα ειδοποίηση χρέωσης και εκτέλεσε `modal deploy llm_service.py` μόνο αν αποδέχεσαι τις χρεώσεις GPU. Όρισε ως `LLM_URL` την ακριβή διεύθυνση προέλευσης HTTPS που εμφανίστηκε κατά τη **δική σου** ανάπτυξη, όχι κάποιον σύνδεσμο που σου έστειλε άλλος. Δώσε στην ιδιωτική διεργασία του πελάτη πρόσβαση στα `API_TOKEN`, `MODAL_KEY` και `MODAL_SECRET` μέσω του διαχειριστή μυστικών. Μην επικολλήσεις μυστικά σε πηγαίο κώδικα, έξοδο, URL ή κοινόχρηστη εντολή κελύφους.
3. Εκτέλεσε `python check_service.py`. Χωρίς κεφαλίδες αναμένεται 401 από τον διαμεσολαβητή· με το ζεύγος διαπιστευτηρίων του αναμένεται 200 από τη `/health`. Το 503 **από την εφαρμογή** σημαίνει ότι απέτυχε ο τοπικός έλεγχος του vLLM. Αν η υπηρεσία παραμένει ανέτοιμη, αναζήτησε σφάλματα εκκίνησης με `modal app logs my-llm-service`. Μην επαναλαμβάνεις συχνά τον έλεγχο: αν τον επιτρέψει ο διαμεσολαβητής, μπορεί να ενεργοποιήσει μια GPU ή να την κρατήσει ενεργή, παρότι δεν παράγει token.
4. Με έγκυρο ζεύγος για τον διαμεσολαβητή αλλά χωρίς έγκυρο bearer για την εφαρμογή, η `/chat` πρέπει να επιστρέψει 401 **από την εφαρμογή**. Με έγκυρα διαπιστευτήρια και στα δύο επίπεδα, ένα κενό μήνυμα πρέπει να λάβει 422 και ένα έγκυρο μήνυμα μη κενή απάντηση. Το χρονικό όριο των 600 δευτερολέπτων αφορά τη συνάρτηση του Modal, όχι τον πελάτη. Ο πελάτης ορίζει δικά του χρονικά όρια και δεν ακολουθεί ανακατευθύνσεις με κεφαλίδες που περιέχουν μυστικά.
5. Όταν τελειώσεις, ακόμη κι αν αποτύχει κάποιος έλεγχος, εκτέλεσε `modal app stop my-llm-service` και βεβαιώσου από τον πίνακα ελέγχου ότι δεν απομένουν ενεργά container. Ανακάλεσε τα διαπιστευτήρια της επίδειξης όταν δεν τα χρειάζεσαι πια.

Οι `/docs` και `/openapi.json` προστατεύονται από τον ίδιο διαμεσολαβητή. Αντί να ανοίξεις απλώς τη διεύθυνση URL, χρησιμοποίησε τον παραπάνω πελάτη που στέλνει κεφαλίδες. Κράτησε την `Authorization` για το διακριτικό **της εφαρμογής** στη `/chat`· μην τοποθετείς ποτέ στην ίδια κεφαλίδα διαπιστευτήρια του διαμεσολαβητή μαζί με δεύτερο διακριτικό bearer.

**Ολοκλήρωση τοπικά (χωρίς ανάπτυξη):** Εκτέλεσε τους ελέγχους του κεφαλαίου 16 με προσομοιωμένες αποκρίσεις HTTP. Βρες ποιο αίτημα του πελάτη που εμφανίζεται παραπάνω πρέπει να απορριφθεί πριν ξεκινήσει το container και ποιο πρέπει να απορρίψει η εφαρμογή.

**Τι δεν αποδεικνύουν οι τοπικοί έλεγχοι:** Ο `TestClient` του FastAPI καλεί απευθείας την εφαρμογή, παρακάμπτοντας τον διαμεσολαβητή του Modal. Μπορεί να ελέγξει τον έλεγχο ταυτότητας της εφαρμογής, τον έλεγχο ετοιμότητας χωρίς παραγωγή κειμένου, την επικύρωση, τα όρια ρυθμού αιτημάτων και τα όρια token. Δεν δείχνει αν ο διαμεσολαβητής του Modal θα δεχτεί πράγματι τα διαπιστευτήρια, πόσο κοστίζει η ψυχρή εκκίνηση ή αν η GPU είναι έτοιμη. Εδώ δεν αναφέρουμε αποτέλεσμα πραγματικής ανάπτυξης ή δοκιμής του διαμεσολαβητή.

## Ενότητα 17.7: Μυστικά και τόμοι αποθήκευσης

### Δημιουργία μυστικών

Στον πίνακα ελέγχου του Modal, δημιούργησε το `my-secrets` με πεδίο `API_KEY` στο περιβάλλον όπου θα εκτελεστεί η συνάρτηση. Αναφέρσου στο μυστικό με το όνομά του· μην ενσωματώσεις την τιμή στον κώδικα και μην την εμφανίσεις. Τα διαπιστευτήρια του διαμεσολαβητή ανήκουν στον έμπιστο πελάτη, όχι στα μυστικά της συνάρτησης.

### Χρήση μυστικών στον κώδικα

In [ ]:
# Χρήση μυστικών στον κώδικα
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Χρησιμοποίησε το κλειδί με ασφάλεια…
'''

print("Χρήση μυστικών:")
print(secrets_example)

In [ ]:
# Αποθήκευση μοντέλων σε τόμους για επαναχρησιμοποίηση
volumes_example = '''
# Δημιούργησε τόμο για την κρυφή μνήμη μοντέλων
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Τα μοντέλα λαμβάνονται μία φορά και αποθηκεύονται στον τόμο για επαναχρησιμοποίηση
    pass
'''

print("Αποθήκευση μοντέλων σε τόμους για επαναχρησιμοποίηση:")
print(volumes_example)
print("\nΣτο πρώτο αίτημα γίνεται λήψη του μοντέλου. Τα επόμενα αιτήματα χρησιμοποιούν το αποθηκευμένο αντίγραφο.")

## Ενότητα 17.8: Επιλογές GPU

| Μέγεθος μοντέλου | Προτεινόμενη GPU | Κόστος ανά ώρα |
|------------|-----------------|----------|
| < 3 δισ. παράμετροι (FP16 ή κβαντισμένο) | T4 (16 GB) | 0,59 $ |
| < 3 δισ. παράμετροι (BF16, π.χ. Qwen3.5-2B) | L4 (24 GB) | 0,80 $ |
| 3–8 δισ. παράμετροι | A10 (24 GB) | 1,10 $ |
| 8–30 δισ. παράμετροι | A100-40GB | 2,50 $ |
| 30 δισ. παράμετροι και άνω | H100 | 3,95 $ |

(Οι τιμές ελέγχθηκαν τον Νοέμβριο του 2026· επιβεβαίωσέ τες στο modal.com/pricing.)

**Ξεκίνα με L4.** Τα βάρη του Qwen3.5-2B είναι BF16. Το vLLM 0.34 δεν δέχεται BF16 σε GPU παλαιότερες από την αρχιτεκτονική Ampere (compute capability 8.0): η T4 έχει έκδοση 7.5 και η εκκίνηση θα αποτύχει, ενώ η L4 (8.9) και η A10 (8.6) καλύπτουν την απαίτηση. Η T4 εξακολουθεί να είναι κατάλληλη για μικρότερα μοντέλα FP16 ή κβαντισμένα μοντέλα. Αναβάθμισε μόνο όταν χρειάζεται.

In [ ]:
# Παραδείγματα επιλογής GPU
print("Η επιλογή GPU γίνεται με μία παράμετρο:")
print()
print('@app.function(gpu="T4")      # 16 GB, 0,59 $/ώρα — μόνο μοντέλα FP16 ή κβαντισμένα (όχι BF16)')
print('@app.function(gpu="L4")      # 24 GB, 0,80 $/ώρα — η επιλογή μας για το Qwen3.5-2B')
print('@app.function(gpu="A10")     # 24 GB, 1,10 $/ώρα — ίδια μνήμη, ταχύτεροι υπολογισμοί')
print('@app.function(gpu="A100")    # Υψηλές επιδόσεις')
print('@app.function(gpu="H100")    # Μέγιστη ισχύς')
print('@app.function(gpu="A100:2")  # Δύο A100')

## Ενότητα 17.9: Διαχείριση της υπηρεσίας μετά την ανάπτυξη

### Προβολή καταγραφών

```bash
modal app logs my-llm-service
```

Εναλλακτικά, χρησιμοποίησε τον πίνακα ελέγχου στο [modal.com](https://modal.com).

### Ενημέρωση της εφαρμογής

```bash
# Ανάπτυξε ξανά την εφαρμογή
modal deploy llm_service.py
```

Το Modal εφαρμόζει αυτόματα την ενημέρωση σταδιακά, ώστε να μη διακοπεί η υπηρεσία.

### Έλεγχος κόστους

- Χρησιμοποίησε τη `scaledown_window` για να σταματούν τα αδρανή container.
- Ξεκίνα με L4 και αναβάθμισε μόνο όταν χρειάζεται.
- Όρισε ειδοποιήσεις δαπανών στον πίνακα ελέγχου του Modal. Με `modal app stop my-llm-service` σταματάς την ανεπτυγμένη εφαρμογή και τα container της.
- Κάθε αίτημα περιορίζεται σε 4.000 χαρακτήρες, 2.048 token εισόδου μετά την εφαρμογή του προτύπου και 512 token εξόδου. Το όριο των 20 αιτημάτων ανά λεπτό και ανά IP προστατεύεται με κλείδωμα, ισχύει **ανά container** και μηδενίζεται όταν το container σταματήσει λόγω μείωσης του φόρτου. Αυτοί οι μηχανισμοί περιορίζουν μεμονωμένα αιτήματα, όχι τη συνολική δαπάνη για πολλά αιτήματα ή αντίγραφα της υπηρεσίας. Σε κοινόχρηστη υπηρεσία, όρισε ποσοστώσεις ανά χρήστη με κοινή κατάσταση που ενημερώνεται ατομικά. Η ειδοποίηση δεν επιβάλλει ανώτατο όριο δαπανών.

## Σύνοψη

Τώρα ξέρεις πώς να:

1. **Αναπτύσσεις συναρτήσεις** στο Modal με `@app.function()`.
2. **Προσθέτεις υποστήριξη GPU** με μία παράμετρο: `gpu="L4"`.
3. **Εγκαθιστάς εξαρτήσεις** με `modal.Image`.
4. **Προστατεύεις όλες τις διαδρομές ιστού** πριν ξεκινήσει το container με `@modal.asgi_app(requires_proxy_auth=True)`.
5. **Αποθηκεύεις μυστικά** με ασφάλεια με `modal.Secret`.
6. **Αποθηκεύεις μοντέλα για επαναχρησιμοποίηση** με `modal.Volume`.
7. **Ελέγχεις το κόστος** με χρονικά όρια αδράνειας και παρακολούθηση.

**Μοιράσου τη διεπαφή, όχι τα διαπιστευτήριά σου.** Η διεύθυνση URL από μόνη της δεν παρέχει πρόσβαση. Κράτησε το ζεύγος του διαμεσολαβητή του χώρου εργασίας σε έμπιστο πελάτη ή εξυπηρετητή. Δώσε το ξεχωριστό διακριτικό bearer μόνο σε όσους έχουν εξουσιοδότηση να παράγουν κείμενο. Για τελικούς χρήστες, βάλε μπροστά από την υπηρεσία δική σου πύλη με έλεγχο ταυτότητας, αντί να διανέμεις διαπιστευτήρια του χώρου εργασίας.

**Εντολές που αξίζει να θυμάσαι:**
```bash
pip install modal      # Εγκατάσταση
modal setup            # Έλεγχος ταυτότητας του CLI
modal run file.py      # Εκτέλεση μία φορά
modal deploy file.py   # Ανάπτυξη μέχρι να σταματήσεις την εφαρμογή
modal app logs name    # Προβολή καταγραφών
modal app stop name    # Διακοπή της ανεπτυγμένης εφαρμογής
```